# 🚀 Quantum Computing Notebook

Algorithms: **Deutsch-Jozsa**, **Grover**, **VQE**  
Libraries: **Qiskit**, **Qiskit Aer**, **PennyLane**  
Compatible with **Google Colab** — run each cell in order.

---
## 1. Install Dependencies

In [ ]:
#@title Install Qiskit + Aer + PennyLane
!pip install -q qiskit qiskit-aer pennylane

---
## 2. Imports

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import pennylane as qml
from pennylane import numpy as pnp
import matplotlib.pyplot as plt
print("All imports successful ✓")

---
## 3. Basic Quantum Circuit with Measurement

Creates a 2-qubit circuit with superposition (Hadamard) and entanglement (CNOT).

In [ ]:
# Build basic circuit
qc = QuantumCircuit(2, 2)
qc.h(0)          # Superposition on qubit 0
qc.cx(0, 1)      # CNOT entangles qubit 0 and 1
qc.measure([0, 1], [0, 1])

print(qc.draw())

# Simulate 1024 shots
sim = AerSimulator()
t_qc = transpile(qc, sim)
result = sim.run(t_qc, shots=1024).result()
counts = result.get_counts()

print(f"\nMeasurement results: {counts}")
plot_histogram(counts)

---
## 4. Deutsch-Jozsa Algorithm

Determines if a boolean function is **constant** (all 0 or all 1) or **balanced** (equal 0s and 1s) using a single query.

In [ ]:
def deutsch_jozsa(oracle_type: str):
    """
    oracle_type: 'constant' or 'balanced'
    Returns 'constant' if result is all-zeros, else 'balanced'.
    """
    n = 3  # number of input qubits
    qc = QuantumCircuit(n + 1, n)

    # Initialize output qubit to |1⟩
    qc.x(n)
    qc.h(range(n + 1))

    # Build oracle
    if oracle_type == 'balanced':
        # CNOTs from each input to output — balanced oracle
        for i in range(n):
            qc.cx(i, n)
    # else 'constant': do nothing (identity)

    qc.h(range(n))
    qc.measure(range(n), range(n))

    sim = AerSimulator()
    t_qc = transpile(qc, sim)
    result = sim.run(t_qc, shots=1).result()
    counts = result.get_counts()

    measured = list(counts.keys())[0]
    return 'balanced' if int(measured, 2) != 0 else 'constant'


# Test both oracles
for oracle in ['constant', 'balanced']:
    res = deutsch_jozsa(oracle)
    print(f"Oracle '{oracle:>8}' → Algorithm says: {res}")
    assert res == oracle, f"Mismatch!"
print("\n✓ Deutsch-Jozsa works correctly!")

---
## 5. Grover's Search Algorithm

Searches an unstructured database of 4 elements (2 qubits) for the marked state |11⟩ — quadratically faster than classical.

In [ ]:
# Grover's algorithm for 2 qubits, searching for |11⟩
n = 2
qc = QuantumCircuit(n, n)

# Step 1: Superposition
qc.h(range(n))

# Step 2: Oracle — marks |11⟩ with a phase flip
qc.cz(0, 1)

# Step 3: Diffusion operator
qc.h(range(n))
qc.x(range(n))
qc.h(1)
qc.cx(0, 1)
qc.h(1)
qc.x(range(n))
qc.h(range(n))

# Step 4: Measure
qc.measure(range(n), range(n))

print(qc.draw())

sim = AerSimulator()
t_qc = transpile(qc, sim)
result = sim.run(t_qc, shots=1024).result()
counts = result.get_counts()

print(f"\nResults: {counts}")
top = max(counts, key=counts.get)
print(f"Most frequent state: |{top}⟩  (target was |11⟩)")
plot_histogram(counts)

---
## 6. VQE — Variational Quantum Eigensolver

Finds the ground-state energy of a simple Hamiltonian (H₂ molecule minimal model) using PennyLane.

In [ ]:
# VQE with PennyLane
# Hamiltonian: H = X0 X1 + 0.5 * Z0 + 0.5 * Z1   (min eigenvalue ~ -1.5)

dev = qml.device('default.qubit', wires=2)

@qml.qnode(dev)
def circuit(params):
    qml.RY(params[0], wires=0)
    qml.RY(params[1], wires=1)
    qml.CNOT(wires=[0, 1])
    qml.RY(params[2], wires=0)
    qml.RY(params[3], wires=1)
    return qml.expval(qml.PauliX(0) @ qml.PauliX(1) + 0.5 * qml.PauliZ(0) + 0.5 * qml.PauliZ(1))

# Classical optimizer (Adam via gradient descent)
opt = qml.AdamOptimizer(stepsize=0.3)
params = pnp.array([0.5, 0.5, 0.5, 0.5], requires_grad=True)

energies = []
for step in range(80):
    params, prev_energy = opt.step_and_cost(circuit, params)
    energies.append(prev_energy)
    if (step + 1) % 20 == 0:
        print(f"Step {step+1:3d}  Energy = {prev_energy:.6f}")

final_energy = circuit(params)
print(f"\n✓ VQE converged! Final energy = {final_energy:.6f}")
print(f"  (Expected ground state ≈ -1.5)")

# Plot convergence
plt.figure(figsize=(6, 3))
plt.plot(energies)
plt.xlabel('Step')
plt.ylabel('Energy')
plt.title('VQE Convergence')
plt.grid(True)
plt.show()

---
## 7. Summary

| Algorithm | Qubits | Result |
|-----------|--------|--------|
| Basic Circuit | 2 | Bell-state |
| Deutsch-Jozsa | 3+1 | Constant / Balanced |
| Grover Search | 2 | Finds marked state |
| VQE | 2 | Ground-state energy |

✅ All cells ready for Google Colab!